# MTM max fixed basis at the extent rule's reach (Fusion_PhD-bhx7.7)

The `cye1` WIDTH ladder kept the extent rule's per-case basis, so $\theta_{first}$ and the reach scaled together. Here the basis is fixed at GFTM's maximum, `NXGRID = 65`, `NBASIS_MAX = NBASIS_MIN = 59` on every case, and only WIDTH varies per case: $\mathrm{WIDTH} = s\,R/x_{max}(65)$ with $R$ the extent rule's one-sided reach (`MTM_analysis.mtm_settings_rule.extent_rule`, ky from the GS2 cube's own coordinate). **Arm A** is $s = 1$ (reach = rule), **Arm B** is $s = 1.41$. $\theta_{first} = \mathrm{WIDTH}\,x_1$ is then as fine as GFTM allows while the reach stays the rule's. Built by GyroRun (`results/mtm_maxbasis/config.yaml`, `code_flags_per_sample`), NSTX_MTM `beta_q_shat_ky_n1000` only.

Compared with `m11_new`, `m9_new`, `extent_rule_tf035` ($s=1$), `extent_rule_tf035_s141` and `gftm_fw174`. Metric as in `mtm_settings_recommendation`: `medlog` = median over GS2-unstable cases ($\gamma_{GS2} > 10^{-3}$) of $|\log_{10}(\gamma_{GFTM}/\gamma_{GS2})|$ on GFTM's **dominant** mode (`argmax` over `mode`); **misses are $+\infty$ and never dropped** (an arm with no deck, or no $\gamma>0$ mode). "Full" = all GS2-unstable cases; "paired" = those where every arm has a deck (the 990 the extent rule could build; Arms A/B also ran the 10 it refused, which therefore count only in "full").

**GS2 tearing** = `general_analysis.mode_classification.classify` == `"MTM"` (Fusion_PhD-7k70, branch `analysis/mode-classification`, calibrated on this very GS2 database). Its `mixed`/`KBM` cases go to "GS2 other". As a sensitivity check the same split is also made with GS2 even fraction $E>0.5$ (the user's 2026-09-30 definition). **Capture** = fraction of GS2-tearing cases whose dominant GFTM mode has even $A_\parallel$ fraction $E>0.5$ (a miss is not captured). GFTM's $E$ is computed here from each cube's own theta-resolved `apar` eigenfunction (unweighted flip about $\theta=0$; GFTM parity is exact, so it is 0 or 1 and the metric weight does not matter), checked against the `apar_even_fraction` the save job stored for Arms A/B.

## Imports and settings

Set `GK_DATA_ROOT` in `local.env`. Comparator arms are leaves beside the new ones.

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv
import numpy as np
import matplotlib.pyplot as plt
from pyrokinetics import Pyro, PyroHypercube
from general_analysis.mode_classification import indicators, classify

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "notebooks").is_dir()
)
plt.style.use(ROOT / "src/general_analysis/paper.mplstyle")
load_dotenv(ROOT / "local.env", override=True)

analysis_name = "mtm_maxbasis"
data_root = Path(os.environ["GK_DATA_ROOT"]).expanduser()
run_template = "Runs"
project = "LATIN_HYPERCUBE"
case = "NSTX_MTM"
draw = "beta_q_shat_ky_n1000"
reference = ("GS2", "pyro_cube_avg")
metadata_file = "pyroscan.json"

# label -> (leaf, folder with the cube, file kind). New arms: GyroRun leaf itself, pyroscan.nc.
arms = {
    "A maxbasis s=1":    ("extent_maxbasis_s100", "", "gyrorun"),
    "B maxbasis s=1.41": ("extent_maxbasis_s141", "", "gyrorun"),
    "m11_new":           ("m11_new", "pyro_cube", "cube"),
    "m9_new":            ("m9_new", "pyro_cube", "cube"),
    "rule s=1":          ("extent_rule_tf035", "pyro_cube", "cube"),
    "rule s=1.41":       ("extent_rule_tf035_s141", "pyro_cube", "cube"),
    "fw174":             ("gftm_fw174", "pyro_cube", "cube"),
}
new_arms = ["A maxbasis s=1", "B maxbasis s=1.41"]
control = "m11_new"
# fixed-geometry arms: (WIDTH, NXGRID) from the registry / Fusion_PhD-lgp2 (fw174 really ran NXGRID 24)
fixed_arms = {"m11_new": (5.89285, 30), "m9_new": (5.41626, 26), "fw174": (1.74, 24)}
nbasis_fixed = {"m11_new": 24, "m9_new": 20, "fw174": 12}

gs2_unstable = 1e-3
tearing_cut = 0.5     # even fraction of A_par
n_boot, seed = 2000, 0

## Load data

Every hypercube through pyrokinetics: the GS2 reference and the GFTM arms.

In [ ]:
gs2_dir = data_root.joinpath(reference[0], run_template, project, case, draw, reference[1])
gs2_cube = PyroHypercube(pyroscan_json=gs2_dir / metadata_file, load_base_pyro=True)
gs2_cube.from_netcdf(gs2_dir / "cube.nc")
gs2 = gs2_cube.gk_output.data

data = {}
for label, (leaf, sub, kind) in arms.items():
    d = data_root.joinpath("GFTM", run_template, project, case, draw, leaf, sub)
    cube = PyroHypercube(pyroscan_json=d / metadata_file, load_base_pyro=True)
    if kind == "gyrorun":
        cube.load_gk_output(netcdf_file=d / "pyroscan.nc")
    else:
        cube.from_netcdf(d / "cube.nc")
    data[label] = cube.gk_output.data
    print(f"{label:18s} samples {data[label].sizes['sample']}, variables {sorted(set(data[label].data_vars) | set(data[label].coords) - {'sample'})[:0] or ''}"
          f"{[v for v in ('apar_even_fraction', 'tearing_parameter', 'nxgrid_used', 'width_used') if v in data[label]]}")

## Calculate: growth-rate error, parity and geometry per case

Arms are matched to GS2 by `sample_name`. A case with no entry, or no mode with $\gamma>0$, scores $+\infty$. $\theta_{first}=\mathrm{WIDTH}\,x_1$ and reach $=\mathrm{WIDTH}\,x_{max}$ use the roots of GFTM's $(2\,\mathrm{NXGRID}-1)$-point Hermite rule (`hermgauss`); for the new arms WIDTH and NXGRID are what `out.gftm.localdump` says GFTM **used**, never `gk_output`'s common output $\theta$ grid.

In [ ]:
names = [str(n) for n in gs2.sample_name.values]
gamma_gs2 = np.asarray(gs2.growth_rate.values, float).reshape(len(names), -1)[:, 0]
unstable = gamma_gs2 > gs2_unstable

def even_fraction(ds):
    # (sample, mode) even fraction of A_par about theta=0; NaN for an empty mode
    a = np.asarray(ds.eigenfunctions.sel(field="apar").transpose("sample", "mode", "theta").values)
    th = np.asarray(ds.theta.values, float)
    r = a[..., ::-1]
    num = lambda f: np.trapezoid(np.abs(f) ** 2, th, axis=-1)
    even, odd = num(a + r), num(a - r)
    with np.errstate(invalid="ignore", divide="ignore"):
        return even / (even + odd)

nodes = lambda n: np.polynomial.hermite.hermgauss(2 * int(n) - 1)[0]
gamma, err, dom_E, have, first, reach, nbasis = {}, {}, {}, {}, {}, {}, {}
for label, ds in data.items():
    row = {str(n): i for i, n in enumerate(ds.sample_name.values)}
    take = np.array([row.get(n, -1) for n in names])
    have[label] = take >= 0
    g = np.asarray(ds.growth_rate.transpose("sample", "mode").values, float)
    gm = np.where(np.isfinite(g), g, -np.inf)
    k = gm.argmax(axis=1)                        # dominant mode, never isel(mode=0)
    E = even_fraction(ds)
    gamma[label] = np.where(have[label], gm.max(axis=1)[np.maximum(take, 0)], np.nan)
    dom_E[label] = np.where(have[label], E[np.arange(len(k)), k][np.maximum(take, 0)], np.nan)
    with np.errstate(divide="ignore", invalid="ignore"):
        e = np.abs(np.log10(gamma[label] / gamma_gs2))
    err[label] = np.where(gamma[label] > 0, e, np.inf)
    if label in fixed_arms:
        w, n = fixed_arms[label]
        first[label] = np.full(len(names), w * nodes(n)[len(nodes(n)) // 2 + 1] / np.pi)
        reach[label] = np.full(len(names), w * nodes(n).max() / np.pi)
        nbasis[label] = np.full(len(names), float(nbasis_fixed[label]))
    else:
        nx, w = ds.nxgrid_used.values.astype(int), ds.width_used.values
        x1 = {n: nodes(n)[len(nodes(n)) // 2 + 1] for n in np.unique(nx)}
        xm = {n: nodes(n).max() for n in np.unique(nx)}
        f = w * np.array([x1[n] for n in nx]) / np.pi
        r_ = w * np.array([xm[n] for n in nx]) / np.pi
        nb = ds.nbasis_used.values.astype(float)
        first[label] = np.where(have[label], f[np.maximum(take, 0)], np.nan)
        reach[label] = np.where(have[label], r_[np.maximum(take, 0)], np.nan)
        nbasis[label] = np.where(have[label], nb[np.maximum(take, 0)], np.nan)
    if label in new_arms:
        stored = np.asarray(ds.apar_even_fraction.transpose("sample", "mode").values, float)
        print(label, "even fraction from eigenfunctions vs save-job apar_even_fraction: max |diff| over finite modes",
              np.nanmax(np.abs(stored - E)), "| finite in one only:", int((np.isfinite(stored) != np.isfinite(E)).sum()))
paired = unstable & np.all([have[l] for l in arms], axis=0)
print(f"GS2-unstable {unstable.sum()} of {len(names)}; paired (every arm has a deck) {paired.sum()}")
print("misses (+inf) among GS2-unstable:", {l: int(np.isinf(err[l][unstable]).sum()) for l in arms})

## Classify the GS2 mode

Each GS2-unstable case is read from its own run directory (geometry and species are per case). `indicators` needs fields and fluxes. Also stored: GS2's even fraction for the sensitivity split.

In [ ]:
from pyrokinetics.diagnostics.field_line import FieldLine
runs = Path(gs2_cube.base_directory)
gs2_class = np.full(len(names), "", dtype=object)
gs2_E = np.full(len(names), np.nan)
for i in np.flatnonzero(unstable):
    pyro = Pyro(gk_file=runs / names[i] / gs2_cube.file_name)
    pyro.load_gk_output()
    gs2_class[i] = classify(indicators(pyro))
    gs2_E[i] = float(FieldLine(pyro).compute_linear_parity().values.ravel()[0])
tear_cls = unstable & (gs2_class == "MTM")
tear_E = unstable & (gs2_E > tearing_cut)
print({c: int((gs2_class[unstable] == c).sum()) for c in ("MTM", "KBM", "mixed")},
      "| E>0.5:", int(tear_E.sum()), "| both:", int((tear_cls & tear_E).sum()), "| classifier MTM but E<=0.5:", int((tear_cls & ~tear_E).sum()))

## Score every arm, with bootstrap intervals

Same resampled cases for every arm inside a population, so differences to `m11_new` are paired. Capture uses the classifier's GS2-tearing cases (E>0.5 variant printed beside it).

In [ ]:
rng = np.random.default_rng(seed)
pops = {"full": unstable, "paired": paired,
        "GS2 tearing (paired)": paired & tear_cls, "GS2 other (paired)": paired & unstable & ~tear_cls,
        "GS2 tearing E>0.5 (paired)": paired & tear_E, "GS2 other E<=0.5 (paired)": paired & unstable & ~tear_E}
table = {}
for pop, m in pops.items():
    b = rng.integers(0, m.sum(), (n_boot, m.sum()))
    for label in arms:
        e = err[label][m]
        cap = (dom_E[label][m] > tearing_cut) & (gamma[label][m] > 0)
        table[pop, label] = dict(n=int(m.sum()), medlog=np.median(e), boot=np.median(e[b], axis=1),
                                 misses=int(np.isinf(e).sum()), capture=cap.mean(), nbasis=np.nanmean(nbasis[label][m]))
    for label in arms:
        r, ref = table[pop, label], table[pop, control]
        r["ci"] = np.percentile(r["boot"], [2.5, 97.5])
        r["delta"] = r["medlog"] - ref["medlog"]
        r["delta_ci"] = np.percentile(r["boot"] - ref["boot"], [2.5, 97.5])
for pop in pops:
    print(f"\n{pop} (n={table[pop, control]['n']}): medlog [95% CI] | minus m11_new [95% CI] | misses | capture | mean NBASIS")
    for label in arms:
        r = table[pop, label]
        print(f"  {label:18s} {r['medlog']:.3f} [{r['ci'][0]:.3f},{r['ci'][1]:.3f}] | {r['delta']:+.3f} "
              f"[{r['delta_ci'][0]:+.3f},{r['delta_ci'][1]:+.3f}] | {r['misses']:3d} | {r['capture']:.3f} | {r['nbasis']:5.1f}")

## Geometry of every arm

Median $\theta_{first}/\pi$ and reach$/\pi$ over the paired cases, per arm.

In [ ]:
print(f"{'arm':18s} theta_first/pi median [q10,q90] | reach/pi median [q10,q90]")
for label in arms:
    f, r = first[label][paired], reach[label][paired]
    print(f"  {label:18s} {np.median(f):.3f} [{np.percentile(f, 10):.3f},{np.percentile(f, 90):.3f}] | "
          f"{np.median(r):.2f} [{np.percentile(r, 10):.2f},{np.percentile(r, 90):.2f}]")

## Plot

Figure 1: $\theta_{first}$ against reach for every arm (paired cases; clouds for the per-case arms, a point for the fixed-geometry arms), legend carries each arm's paired medlog. This is the separation the `cye1` ladder could not make: the rule and the ladder move along the diagonal, the max-basis arms sit at the floor of $\theta_{first}$ for their reach. Figure 2: medlog and capture per arm with 95% bootstrap bars for the four populations.

In [ ]:
fig1, ax = plt.subplots(figsize=(7.5, 5.5))
colours = {l: f"C{i}" for i, l in enumerate(arms)}
for label in arms:
    f, r = first[label][paired], reach[label][paired]
    m = table["paired", label]["medlog"]
    name = f"{label} (medlog {m:.3f})"
    if label in fixed_arms:
        ax.plot(r[0], f[0], "*", ms=14, color=colours[label], label=name)
    else:
        ax.plot(r, f, ".", ms=3, alpha=0.35, color=colours[label])
        ax.plot(np.median(r), np.median(f), "o", ms=9, mec="k", color=colours[label], label=name)
ax.set(xscale="log", yscale="log", xlabel=r"reach $= \mathrm{WIDTH}\,x_{max}$ [$\pi$]", ylabel=r"$\theta_{first} = \mathrm{WIDTH}\,x_1$ [$\pi$]",
       title="Quadrature geometry of each arm, NSTX_MTM n1000 (paired cases)")
ax.legend(fontsize=8)
plt.show()

In [ ]:
shown = ["full", "paired", "GS2 tearing (paired)", "GS2 other (paired)"]
fig2, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
x = np.arange(len(arms))
for j, pop in enumerate(shown):
    rs = [table[pop, l] for l in arms]
    axes[0].errorbar(x + 0.12 * (j - 1.5), [r["medlog"] for r in rs],
                     yerr=np.array([np.abs(r["ci"] - r["medlog"]) for r in rs]).T, ls="", marker="o", label=f"{pop} (n={rs[0]['n']})")
axes[1].bar(x, [table["GS2 tearing (paired)", l]["capture"] for l in arms])
axes[0].set(ylabel="medlog (misses kept)")
axes[1].set(ylabel="capture of GS2-tearing cases", xticks=x, xticklabels=list(arms), ylim=(0, 1))
axes[0].legend(fontsize=8)
plt.setp(axes[1].get_xticklabels(), rotation=25, ha="right")
plt.show()

## Save

Replaces the same filenames.

In [ ]:
output_dir = ROOT / "Plots" / analysis_name
output_dir.mkdir(parents=True, exist_ok=True)
fig1.savefig(output_dir / "fig1_theta_first_vs_reach.png")
fig2.savefig(output_dir / "fig2_medlog_capture.png")

## Interpretation

_To be written after the executed run._